# 使用百度千帆智能搜索生成API获取最新技术资讯

本笔记本演示如何使用百度千帆智能搜索生成(AI Search)功能，获取感兴趣领域的最新资讯。

**参考文档**: 
- [智能搜索生成API文档](https://cloud.baidu.com/doc/AppBuilder/s/zm8pn5cju)
- [百度AI搜索](https://ai.baidu.com/ai-doc/AppBuilder/amaxd2det)

**特点**:
- ✅ 无需创建应用，直接使用API Key
- ✅ 支持实时联网搜索
- ✅ AI自动总结搜索结果
- ✅ 每日免费100次调用

## 步骤1: 导入必要的库

In [ ]:
import json
import requests
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import re
from typing import List, Dict, Any, Optional

print("✓ 库导入成功")

## 步骤2: 加载环境变量和配置

从 `.env` 文件中加载API密钥（无需应用ID）

In [ ]:
# 加载环境变量
load_dotenv()

# 获取API凭证（只需要API Key，不需要App ID）
QIANFAN_API_KEY = os.getenv('QIANFAN_API_KEY')  # AppBuilder API Key

# API配置
API_URL = "https://qianfan.baidubce.com/v2/ai_search/chat/completions"

# 验证API密钥是否加载成功
if QIANFAN_API_KEY:
    print(f"✓ API密钥已加载 (长度: {len(QIANFAN_API_KEY)} 字符)")
    print(f"✓ API端点: {API_URL}")
else:
    print("✗ 警告: API密钥未找到，请检查.env文件")
    print("  请在.env文件中添加: QIANFAN_API_KEY=your_api_key_here")

## 步骤3: 定义感兴趣的技术领域

自定义你想要获取资讯的技术领域和时间范围

In [ ]:
# 定义感兴趣的技术领域
TECH_FIELDS = [
    "自动驾驶",
    "具身智能",
    "大模型",
    "人工智能"
]

# 定义重点关注的主题
FOCUS_TOPICS = [
    "特斯拉FSD",
    "人形机器人",
    "AI大模型突破",
    "行业重要动态"
]

# 时间范围设置
# 可选: 'week' (最近7天), 'month' (最近30天), 'semiyear' (最近180天), 'year' (最近365天)
SEARCH_RECENCY = "week"  # 最近7天

# 使用的模型
MODEL = "deepseek-r1"  # 推荐: deepseek-r1 (支持推理), ernie-3.5-8k, ernie-4.5-turbo-32k 等

# 搜索配置
ENABLE_DEEP_SEARCH = False  # 是否开启深度搜索（会消耗更多次数）
ENABLE_REASONING = True  # 是否开启推理模式（deepseek-r1支持）
MAX_SEARCH_QUERY_NUM = 10  # 最大搜索查询数量
MAX_COMPLETION_TOKENS = 20480  # 最大生成token数

# 资源类型配置（每种类型返回的结果数量）
RESOURCE_CONFIG = {
    "image": 4,  # 图片
    "video": 4,  # 视频
    "web": 4     # 网页
}

print(f"关注领域: {', '.join(TECH_FIELDS)}")
print(f"重点主题: {', '.join(FOCUS_TOPICS)}")
print(f"时间范围: {SEARCH_RECENCY}")
print(f"使用模型: {MODEL}")
print(f"推理模式: {'开启' if ENABLE_REASONING else '关闭'}")
print(f"深度搜索: {'开启' if ENABLE_DEEP_SEARCH else '关闭'}")
print(f"资源类型: {list(RESOURCE_CONFIG.keys())}")

## 步骤4: 创建AI搜索客户端

封装百度千帆智能搜索API

In [ ]:
class QianfanAISearchClient:
    """
    百度千帆智能搜索生成客户端
    """
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.api_url = "https://qianfan.baidubce.com/v2/ai_search/chat/completions"
        
    def _get_headers(self) -> Dict[str, str]:
        """获取请求头"""
        return {
            'X-Appbuilder-Authorization': f'Bearer {self.api_key}',
            'Content-Type': 'application/json'
        }
    
    def search(self, 
               query: str,
               model: str = "deepseek-r1",
               search_recency: str = None,
               enable_deep_search: bool = False,
               enable_reasoning: bool = True,
               resource_config: Dict[str, int] = None,
               max_search_query_num: int = 10,
               max_completion_tokens: int = 20480,
               previous_response: str = None,
               stream: bool = False) -> Optional[Dict[str, Any]]:
        """
        执行AI搜索
        
        参数:
            query: 搜索查询
            model: 使用的模型（如 deepseek-r1, ernie-3.5-8k）
            search_recency: 时间范围过滤 ('week', 'month', 'semiyear', 'year')
            enable_deep_search: 是否开启深度搜索
            enable_reasoning: 是否开启推理模式（deepseek-r1支持）
            resource_config: 资源类型配置，如 {"web": 4, "image": 4, "video": 4}
            max_search_query_num: 最大搜索查询数量
            max_completion_tokens: 最大生成token数
            previous_response: 上一次的助手响应（用于多轮对话）
            stream: 是否流式输出
        
        返回:
            搜索结果
        """
        headers = self._get_headers()
        
        # 默认资源配置
        if resource_config is None:
            resource_config = {"web": 10}
        
        # 构建消息列表（支持多轮对话）
        messages = [
            {
                "role": "user",
                "content": query
            }
        ]
        
        # 如果有上一次的响应，添加到消息历史中
        if previous_response:
            messages.append({
                "role": "assistant",
                "content": previous_response
            })
        
        # 构建资源类型过滤器
        resource_type_filter = []
        for res_type, top_k in resource_config.items():
            resource_type_filter.append({
                "type": res_type,
                "top_k": top_k
            })
        
        # 构建请求体（使用与demo相同的格式）
        body = {
            "messages": messages,
            "model": model,
            "stream": stream,
            "temperature": "1e-10",  # 极低温度，获得确定性输出
            "top_p": "1e-10",  # 极低top_p，与温度配合使用
            "search_source": "baidu_search_v2",  # 使用V2版本，性能更好
            "resource_type_filter": resource_type_filter,
            "enable_deep_search": enable_deep_search,
            "enable_corner_markers": True,  # 开启角标
            "enable_followup_queries": False,  # 不需要追问
            "search_mode": "required",  # 必须执行搜索
            "max_search_query_num": max_search_query_num,
            "max_completion_tokens": str(max_completion_tokens)  # 转为字符串
        }
        
        # deepseek-r1 模型支持推理模式
        if "deepseek" in model.lower() and enable_reasoning:
            body["enable_reasoning"] = True
        
        # 添加时间范围过滤
        if search_recency:
            body["search_recency_filter"] = search_recency
        
        try:
            response = requests.post(self.api_url, json=body, headers=headers, timeout=120)
            
            if response.status_code == 200:
                if stream:
                    return self._handle_stream_response(response)
                else:
                    return response.json()
            else:
                print(f"✗ API调用失败: {response.status_code}")
                print(f"响应: {response.text}")
                return None
        except Exception as e:
            print(f"✗ API调用错误: {e}")
            return None
    
    def _handle_stream_response(self, response) -> Dict[str, Any]:
        """
        处理流式响应
        """
        full_content = ""
        last_data = None
        
        for line in response.iter_lines():
            if line:
                line_str = line.decode('utf-8')
                if line_str.startswith('data: '):
                    try:
                        data = json.loads(line_str[6:])
                        last_data = data
                        
                        if 'choices' in data and len(data['choices']) > 0:
                            delta = data['choices'][0].get('delta', {})
                            content = delta.get('content', '')
                            if content:
                                full_content += content
                    except json.JSONDecodeError:
                        pass
        
        # 返回汇总的响应
        if last_data and 'choices' in last_data:
            last_data['choices'][0]['message'] = {
                'content': full_content,
                'role': 'assistant'
            }
            return last_data
        
        return None

print("✓ QianfanAISearchClient类已定义")

## 步骤5: 创建智能查询prompt

构建结构化的查询prompt

In [ ]:
def create_search_query(fields: List[str], topics: List[str], 
                       time_desc: str = "最近一天") -> str:
    """
    创建搜索查询
    
    参数:
        fields: 技术领域列表
        topics: 关注主题列表
        time_desc: 时间描述
    
    返回:
        查询字符串
    """
    all_topics = fields + topics
    topics_str = ", ".join(all_topics)
    
    # 使用与demo相同的提示词格式
    query = f"请搜索{time_desc}关于以下技术领域的最新动态和重要新闻：{topics_str}。"
    
    return query

print("✓ 查询prompt函数已定义")

## 步骤6: 执行智能搜索

使用千帆AI搜索API获取最新资讯

In [ ]:
# 初始化客户端
if QIANFAN_API_KEY:
    client = QianfanAISearchClient(QIANFAN_API_KEY)
    print("✓ 千帆AI搜索客户端初始化成功")
    
    # 创建查询
    time_desc_map = {
        'week': '最近一周',
        'month': '最近一个月',
        'semiyear': '最近半年',
        'year': '最近一年'
    }
    time_desc = time_desc_map.get(SEARCH_RECENCY, '最近一周')
    
    query = create_search_query(TECH_FIELDS, FOCUS_TOPICS, time_desc)
    
    print(f"\n{'='*80}")
    print("正在执行AI智能搜索...")
    print(f"{'='*80}\n")
    print(f"查询内容: {query}\n")
    
    # 执行搜索（使用与demo相同的配置）
    search_result = client.search(
        query=query,
        model=MODEL,
        search_recency=SEARCH_RECENCY,
        enable_deep_search=ENABLE_DEEP_SEARCH,
        enable_reasoning=ENABLE_REASONING,
        resource_config=RESOURCE_CONFIG,
        max_search_query_num=MAX_SEARCH_QUERY_NUM,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        stream=False
    )
    
    if search_result:
        print("\n✓ 搜索完成")
        print(f"  - Request ID: {search_result.get('request_id', 'N/A')}")
        print(f"  - 安全检查: {'通过' if search_result.get('is_safe', True) else '未通过'}")
        
        # 显示token使用情况
        usage = search_result.get('usage', {})
        if usage:
            print(f"  - Token使用: {usage.get('total_tokens', 0)} (输入: {usage.get('prompt_tokens', 0)}, 输出: {usage.get('completion_tokens', 0)})")
    else:
        print("\n✗ 搜索失败")
else:
    print("✗ 缺少API密钥，无法初始化客户端")
    client = None
    search_result = None

## 步骤7: 展示搜索结果和引用

显示AI总结的内容和搜索引用来源

In [ ]:
def display_search_results(result: Dict[str, Any]):
    """
    显示搜索结果
    """
    if not result:
        print("✗ 无搜索结果")
        return
    
    # 显示AI总结的内容
    choices = result.get('choices', [])
    if choices:
        message = choices[0].get('message', {})
        content = message.get('content', '')
        
        print("\n" + "="*80)
        print(f"{'AI总结内容':^76}")
        print("="*80)
        print(f"\n{content}\n")
        print("-"*80)
    
    # 显示引用来源
    references = result.get('references', [])
    if references:
        print(f"\n📚 引用来源 (共{len(references)}条):\n")
        
        for i, ref in enumerate(references, 1):
            ref_id = ref.get('id', i)
            title = ref.get('title', 'N/A')
            url = ref.get('url', 'N/A')
            website = ref.get('website', '')
            date = ref.get('date', 'N/A')
            ref_type = ref.get('type', 'web')
            snippet = ref.get('content', '')[:150]  # 显示前150字符
            
            type_icon = {'web': '🌐', 'image': '🖼️', 'video': '🎬'}.get(ref_type, '📄')
            
            print(f"\n[{ref_id}] {type_icon} {title}")
            if snippet:
                print(f"   摘要: {snippet}...")
            print(f"   来源: {website if website else url}")
            print(f"   日期: {date}")
            print(f"   链接: {url}")
            print("-"*80)

# 显示结果
if search_result:
    display_search_results(search_result)

## 步骤8: 保存搜索结果

将搜索结果保存为JSON文件

In [ ]:
def save_search_result(result: Dict[str, Any], filename: str = None) -> Optional[str]:
    """
    保存搜索结果到JSON文件
    
    参数:
        result: 搜索结果
        filename: 文件名（可选）
    
    返回:
        保存的文件路径
    """
    if not result:
        print("✗ 无结果可保存")
        return None
    
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"tech_news_qianfan_aisearch_{timestamp}.json"
    
    try:
        # 保存完整的搜索结果
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
        file_size = os.path.getsize(filename)
        print(f"\n✓ 结果已保存到: {filename}")
        print(f"  文件大小: {file_size} 字节")
        
        return filename
    except Exception as e:
        print(f"✗ 保存文件失败: {e}")
        return None

# 保存结果
if search_result:
    saved_file = save_search_result(search_result)
    if saved_file:
        print(f"\n可以使用以下代码重新加载数据:")
        print(f"with open('{saved_file}', 'r', encoding='utf-8') as f:")
        print(f"    result = json.load(f)")

## 步骤9: 数据分析统计

对搜索结果和引用来源进行统计分析

In [ ]:
def analyze_search_result(result: Dict[str, Any]):
    """
    分析搜索结果
    """
    if not result:
        print("无法分析：结果为空")
        return
    
    print("\n" + "="*80)
    print(f"{'搜索结果统计分析':^76}")
    print("="*80)
    
    # 1. 引用来源统计
    references = result.get('references', [])
    
    if references:
        print(f"\n📊 引用来源统计:")
        print(f"   总引用数: {len(references)} 条")
        
        # 按类型统计
        type_count = {}
        for ref in references:
            ref_type = ref.get('type', 'web')
            type_count[ref_type] = type_count.get(ref_type, 0) + 1
        
        print("\n   按类型分布:")
        for ref_type, count in sorted(type_count.items(), key=lambda x: x[1], reverse=True):
            type_name = {'web': '网页', 'image': '图片', 'video': '视频'}.get(ref_type, ref_type)
            print(f"     {type_name}: {count} 条")
        
        # 按网站统计
        website_count = {}
        for ref in references:
            website = ref.get('website', '未知')
            if website:
                website_count[website] = website_count.get(website, 0) + 1
        
        if website_count:
            print("\n   热门信息源 (Top 5):")
            top_websites = sorted(website_count.items(), key=lambda x: x[1], reverse=True)[:5]
            for website, count in top_websites:
                print(f"     {website}: {count} 条")
        
        # 时间分布
        date_count = {}
        for ref in references:
            date = ref.get('date', '')
            if date and date != 'N/A':
                date_count[date] = date_count.get(date, 0) + 1
        
        if date_count:
            print("\n   时间分布 (Top 5):")
            top_dates = sorted(date_count.items(), key=lambda x: x[1], reverse=True)[:5]
            for date, count in top_dates:
                print(f"     {date}: {count} 条")
    
    # 2. Token使用统计
    usage = result.get('usage', {})
    if usage:
        print("\n💰 Token使用统计:")
        print(f"   输入Token: {usage.get('prompt_tokens', 0)}")
        print(f"   输出Token: {usage.get('completion_tokens', 0)}")
        print(f"   总Token: {usage.get('total_tokens', 0)}")
    
    # 3. 内容长度统计
    choices = result.get('choices', [])
    if choices:
        content = choices[0].get('message', {}).get('content', '')
        print("\n📝 生成内容统计:")
        print(f"   内容长度: {len(content)} 字符")
        print(f"   估计字数: {len(content) // 2} 字（中文）")
    
    print("\n" + "="*80)

# 执行分析
if search_result:
    analyze_search_result(search_result)

## 步骤10: 完整流程封装

将以上步骤封装成一个完整的函数

In [ ]:
def acquire_latest_news_qianfan(
    api_key: str,
    fields: List[str],
    focus_topics: List[str],
    search_recency: str = "week",
    model: str = "deepseek-r1",
    enable_deep_search: bool = False,
    enable_reasoning: bool = True,
    resource_config: Dict[str, int] = None,
    max_search_query_num: int = 10,
    max_completion_tokens: int = 20480,
    save_to_file: bool = True,
    show_analysis: bool = True
) -> Optional[Dict[str, Any]]:
    """
    完整的新闻获取流程（使用百度千帆AI搜索）
    
    参数:
        api_key: 千帆AppBuilder API Key
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        search_recency: 时间范围 ('week', 'month', 'semiyear', 'year')
        model: 使用的模型（推荐 deepseek-r1）
        enable_deep_search: 是否开启深度搜索
        enable_reasoning: 是否开启推理模式（deepseek-r1支持）
        resource_config: 资源类型配置，如 {"web": 4, "image": 4, "video": 4}
        max_search_query_num: 最大搜索查询数量
        max_completion_tokens: 最大生成token数
        save_to_file: 是否保存到文件
        show_analysis: 是否显示统计分析
    
    返回:
        搜索结果
    """
    print("\n" + "="*80)
    print(f"{'开始获取最新技术资讯 (百度千帆AI搜索)':^76}")
    print("="*80)
    
    # 默认资源配置
    if resource_config is None:
        resource_config = {"image": 4, "video": 4, "web": 4}
    
    # 1. 初始化客户端
    print("\n[1/4] 初始化AI搜索客户端...")
    client = QianfanAISearchClient(api_key)
    
    # 2. 创建查询
    print("\n[2/4] 构建搜索查询...")
    time_desc_map = {
        'week': '最近一周',
        'month': '最近一个月',
        'semiyear': '最近半年',
        'year': '最近一年'
    }
    time_desc = time_desc_map.get(search_recency, '最近一周')
    query = create_search_query(fields, focus_topics, time_desc)
    
    # 3. 执行搜索
    print("\n[3/4] 执行AI智能搜索...")
    result = client.search(
        query=query,
        model=model,
        search_recency=search_recency,
        enable_deep_search=enable_deep_search,
        enable_reasoning=enable_reasoning,
        resource_config=resource_config,
        max_search_query_num=max_search_query_num,
        max_completion_tokens=max_completion_tokens,
        stream=False
    )
    
    if not result:
        print("\n✗ 流程终止：搜索失败")
        return None
    
    # 4. 显示结果
    print("\n[4/4] 显示搜索结果...")
    display_search_results(result)
    
    # 5. 保存文件
    if save_to_file:
        save_search_result(result)
    
    # 6. 统计分析
    if show_analysis:
        analyze_search_result(result)
    
    print("\n" + "="*80)
    print(f"{'流程完成！':^76}")
    print("="*80)
    
    return result

print("✓ 完整流程函数已定义")
print("\n快速使用示例:")
print("""
# 获取最新技术资讯（使用与demo相同的配置）
result = acquire_latest_news_qianfan(
    api_key=QIANFAN_API_KEY,
    fields=["自动驾驶", "具身智能", "大模型"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    search_recency="week",  # 最近一周
    model="deepseek-r1",  # 使用deepseek-r1模型
    enable_reasoning=True,  # 开启推理模式
    resource_config={"image": 4, "video": 4, "web": 4},
    max_search_query_num=10,
    max_completion_tokens=20480,
    save_to_file=True,
    show_analysis=True
)
""")

## 总结

本笔记本演示了如何使用百度千帆智能搜索生成API获取最新技术资讯，包括：

1. ✅ 环境配置和API密钥加载（无需应用ID）
2. ✅ 自定义技术领域和关注主题
3. ✅ 创建AI搜索客户端
4. ✅ 构建智能搜索查询
5. ✅ 执行实时联网搜索
6. ✅ 展示AI总结和引用来源
7. ✅ 保存搜索结果
8. ✅ 数据统计分析
9. ✅ 完整流程封装

### 核心优势

#### 1. 简单易用
- **无需创建应用**: 只需API Key即可使用
- **免费额度**: 每日免费100次调用
- **即时可用**: 无需配置应用，开箱即用

#### 2. 强大功能
- **实时联网搜索**: 使用百度搜索引擎V2版本
- **AI智能总结**: 自动整理和总结搜索结果
- **时间过滤**: 支持week/month/semiyear/year
- **深度搜索**: 可选开启深度搜索（最多100个结果）
- **引用标注**: 提供完整的引用来源和角标
- **推理模式**: DeepSeek-R1模型支持推理链展示

#### 3. 丰富配置
- **多模型支持**: deepseek-r1, ernie-3.5-8k, ernie-4.5-turbo等
- **搜索源版本**: baidu_search_v2（推荐）
- **资源类型**: web/image/video（多类型并行搜索）
- **确定性输出**: temperature=1e-10, top_p=1e-10
- **结果数量**: 可自定义每种资源类型的top_k

### 使用建议

#### 获取API Key

1. 访问 [百度智能云控制台](https://console.bce.baidu.com/ai_apaas/resource)
2. 点击侧边栏 **API Key**
3. 点击 **创建API Key**
4. 服务选择 **千帆AppBuilder**
5. 配置权限策略，点击 **确定**
6. 复制生成的API Key

#### 配置示例

在 `.env` 文件中添加：
```bash
# 百度千帆AppBuilder
QIANFAN_API_KEY=bce-v3/ALTAK****/051c6****
```

#### 基础用法（推荐使用与demo相同的配置）

```python
# 使用deepseek-r1模型获取最新资讯
result = acquire_latest_news_qianfan(
    api_key=QIANFAN_API_KEY,
    fields=["自动驾驶", "具身智能", "大模型"],
    focus_topics=["特斯拉FSD", "人形机器人"],
    search_recency="week",  # 最近一周
    model="deepseek-r1",  # 推荐使用deepseek-r1
    enable_reasoning=True,  # 开启推理模式
    resource_config={"image": 4, "video": 4, "web": 4},
    max_search_query_num=10,
    max_completion_tokens=20480
)
```

### 参数说明

| 参数 | 类型 | 说明 | 默认值 |
|------|------|------|--------|
| api_key | str | AppBuilder API Key | 必填 |
| fields | List[str] | 技术领域列表 | 必填 |
| focus_topics | List[str] | 重点主题列表 | 必填 |
| search_recency | str | 时间范围 | "week" |
| model | str | 使用的模型 | "deepseek-r1" |
| enable_reasoning | bool | 是否开启推理模式 | True |
| resource_config | Dict | 资源类型配置 | {"image":4,"video":4,"web":4} |
| max_search_query_num | int | 最大搜索查询数 | 10 |
| max_completion_tokens | int | 最大生成token数 | 20480 |
| enable_deep_search | bool | 是否开启深度搜索 | False |
| save_to_file | bool | 是否保存文件 | True |
| show_analysis | bool | 是否显示分析 | True |

### 时间范围选项

- `week`: 最近7天（推荐用于获取最新动态）
- `month`: 最近30天
- `semiyear`: 最近180天
- `year`: 最近365天

### 支持的模型

- `deepseek-r1`: DeepSeek R1（推荐，支持推理模式，输出质量高）
- `ernie-3.5-8k`: 文心3.5（快速，成本低）
- `ernie-4.5-turbo-32k`: 文心4.5（性能好）
- `deepseek-v3`: DeepSeek V3
- `qwen3-235b-a22b-instruct-2507`: 通义千问
- 更多模型见[官方文档](https://cloud.baidu.com/doc/WENXINWORKSHOP/s/Fm2vrveyu)

### 资源类型配置

可以同时搜索多种类型的资源：

```python
resource_config = {
    "web": 4,    # 网页资源，返回4条
    "image": 4,  # 图片资源，返回4条
    "video": 4   # 视频资源，返回4条
}
```

### DeepSeek-R1 推理模式

使用 `deepseek-r1` 模型时，可以开启 `enable_reasoning=True`：
- 模型会展示推理过程
- 提供更详细的分析思路
- 输出质量更高
- 适合需要深度分析的场景

### 确定性输出

设置 `temperature="1e-10"` 和 `top_p="1e-10"`：
- 获得高度确定性的输出
- 每次相同查询返回相似结果
- 适合需要稳定输出的场景

### 三种方案对比

| 特性 | 千帆AI搜索 | 腾讯混元 | 秘塔META |
|------|-----------|----------|----------|
| 搜索方式 | AI+百度搜索 | AI联网搜索 | 直接搜索引擎 |
| 配置复杂度 | 低（仅API Key） | 中 | 低 |
| 免费额度 | 100次/天 | 无 | 有限 |
| 搜索质量 | 高（百度搜索） | 高 | 中 |
| AI总结 | 是 | 是 | 否 |
| 推理模式 | 支持(DeepSeek-R1) | 不支持 | 不支持 |
| 引用来源 | 完整 | 有 | 完整 |
| 时间过滤 | 原生支持 | AI理解 | 查询词 |
| 深度搜索 | 支持 | 不支持 | 不支持 |
| 资源类型 | web/image/video | 通用 | web/blog/scholar |
| 确定性输出 | 支持(1e-10) | 不支持 | 不支持 |
| 适用场景 | 需要权威来源+AI总结+推理 | 快速AI分析 | 原始信息采集 |

### 注意事项

1. **调用限制**
   - 免费额度：100次/天
   - 深度搜索：会产生10次以内的额外调用
   - 建议关注调用次数，避免超额

2. **成本控制**
   - deepseek-r1推理模式会消耗更多token
   - 深度搜索会增加调用次数
   - 合理设置max_completion_tokens和resource_config

3. **数据质量**
   - AI总结可能存在误差
   - 重要信息建议核对原文
   - 参考引用来源的权威性
   - DeepSeek-R1推理链可以帮助理解AI的分析过程

4. **时间范围**
   - week适合获取最新动态
   - month适合了解近期趋势
   - 时间范围越短，结果越精准

### 最佳实践

1. **优化查询**
   - 使用具体的关键词
   - 明确时间要求
   - 使用简洁的提示词（与demo相同）

2. **结果验证**
   - 检查引用来源的可信度
   - 对比多个来源的信息
   - 访问原文链接核实细节
   - 查看DeepSeek-R1的推理过程理解结论

3. **效率提升**
   - 使用合适的时间范围
   - 避免过于宽泛的查询
   - 定期清理旧的搜索结果
   - 选择合适的模型（deepseek-r1 vs ernie-3.5-8k）

### 下一步扩展

- ✅ 添加定时任务，自动获取资讯
- ✅ 实现多次搜索结果的对比分析
- ✅ 集成通知功能（邮件/微信）
- ✅ 构建个性化推荐系统
- ✅ 添加情感分析和趋势预测
- ✅ 与其他数据源整合对比
- ✅ 利用推理模式进行深度分析